# Experiments 12,13,14
## Date: 23-11-2026

 # 12.Set and Dictionary Operations

In [1]:
from collections import Counter, defaultdict 
  
print("University Course Enrolment Analysis: Sets and Dictionaries") 
print("=" * 60) 
  
# ---------- Sets ---------- 
python_course = {"Anu", "Bala", "Chitra", "Dinesh", "Esha", "Farhan"} 
ml_course = {"Chitra", "Dinesh", "Gokul", "Harini", "Esha"} 
big_data = {"Bala", "Esha", "Harini", "Ishaan"} 
  
print("\nPython course :", sorted(python_course)) 
print("ML course     :", sorted(ml_course)) 
print("Big Data      :", sorted(big_data)) 
print( 
    "\nUnion (all students)          :", 
    sorted(python_course | ml_course | big_data), 
) 
print("Intersection (Python & ML)    :", sorted(python_course & ml_course)) 
print("Difference (Python only, not ML):", sorted(python_course - ml_course)) 
print("Symmetric difference (Py ^ ML):", sorted(python_course ^ ml_course)) 
print( 
    "In all three courses          :", 
    sorted(python_course & ml_course & big_data), 
) 
print( 
    "Is {'Esha','Bala'} subset of Python? :", 
    {"Esha", "Bala"}.issubset(python_course), 
) 
print("Are ML and Big Data disjoint?        :", ml_course.isdisjoint(big_data)) 
  
ml_course.add("Jaya") 
ml_course.discard("Gokul") 
print("ML after add('Jaya'), discard('Gokul'):", sorted(ml_course)) 
duplicates = ["Anu", "Bala", "Anu", "Chitra", "Bala", "Anu"] 
print("Unique names from a list with duplicates:", sorted(set(duplicates))) 
  
# ---------- Dictionaries ---------- 
marks = { 
    "Anu": {"Python": 88, "ML": None}, 
    "Bala": {"Python": 72, "BigData": 81}, 
    "Chitra": {"Python": 95, "ML": 91}, 
    "Dinesh": {"Python": 64, "ML": 70}, 
} 
marks["Anu"]["ML"] = 79  # update nested value 
marks["Esha"] = {"Python": 85, "ML": 88, "BigData": 90}  # insert 
print("\nStudent marks dictionary:") 
for student, sub in marks.items(): 
    print(f"  {student:<7} {sub}") 
  
print("\nKeys   :", list(marks.keys())) 
print("Chitra's Python mark   :", marks["Chitra"]["Python"]) 
print("get() for missing key  :", marks.get("Zara", "Not enrolled")) 
averages = { 
    s: round(sum(v.values()) / len(v), 2) for s, v in marks.items() 
}  # dict comprehension 
print("Average per student    :", averages) 
topper = max(averages, key=averages.get) 
print("Topper                 :", topper, averages[topper]) 
  
subject_scores = defaultdict(list) 
for sub in marks.values(): 
    for subject, m in sub.items(): 
        subject_scores[subject].append(m) 
print("\nSubject-wise statistics:") 
for subject, scores in subject_scores.items(): 
    print( 
        f"  {subject:<8} count={len(scores)}  " 
        f"avg={sum(scores)/len(scores):.2f}  max={max(scores)}" 
    ) 
  
removed = marks.pop("Dinesh") 
print("\nRemoved Dinesh ->", removed, "| students left:", len(marks)) 
marks.update({"Farhan": {"Python": 77}}) 
print("After update() with Farhan:", sorted(marks)) 
  
feedback = ( 
    "good course good faculty excellent labs good projects excellent support" 
) 
freq = Counter(feedback.split()) 
print("\nWord frequency of feedback:", dict(freq)) 
print("Most common 2 words      :", freq.most_common(2)) 
print( 
    "Sorted by value (desc)   :", 
    dict(sorted(freq.items(), key=lambda kv: kv[1], reverse=True)), 
) 

University Course Enrolment Analysis: Sets and Dictionaries

Python course : ['Anu', 'Bala', 'Chitra', 'Dinesh', 'Esha', 'Farhan']
ML course     : ['Chitra', 'Dinesh', 'Esha', 'Gokul', 'Harini']
Big Data      : ['Bala', 'Esha', 'Harini', 'Ishaan']

Union (all students)          : ['Anu', 'Bala', 'Chitra', 'Dinesh', 'Esha', 'Farhan', 'Gokul', 'Harini', 'Ishaan']
Intersection (Python & ML)    : ['Chitra', 'Dinesh', 'Esha']
Difference (Python only, not ML): ['Anu', 'Bala', 'Farhan']
Symmetric difference (Py ^ ML): ['Anu', 'Bala', 'Farhan', 'Gokul', 'Harini']
In all three courses          : ['Esha']
Is {'Esha','Bala'} subset of Python? : True
Are ML and Big Data disjoint?        : False
ML after add('Jaya'), discard('Gokul'): ['Chitra', 'Dinesh', 'Esha', 'Harini', 'Jaya']
Unique names from a list with duplicates: ['Anu', 'Bala', 'Chitra']

Student marks dictionary:
  Anu     {'Python': 88, 'ML': 79}
  Bala    {'Python': 72, 'BigData': 81}
  Chitra  {'Python': 95, 'ML': 91}
  Dinesh  {'Pyth

# 13.Distributed File System and Loading Data into HDFS

In [2]:
import hashlib 
import itertools 
  
BLOCK_SIZE = 64  # bytes (scaled down from HDFS default of 128 MB) 
REPLICATION = 3 
  
  
class DataNode: 
    def __init__(self, name, rack): 
        self.name, self.rack, self.alive = name, rack, True 
        self.blocks = {} 
  
  
class NameNode: 
    """Keeps only metadata: file -> blocks, block -> DataNodes.""" 
  
    def __init__(self, datanodes): 
        self.datanodes = datanodes 
        self.namespace = {}  # path -> list of block ids 
        self.block_map = {}  # block id -> list of DataNode names 
        self._rr = itertools.cycle(range(len(datanodes))) 
        self._next_id = 1 
  
    def choose_targets(self): 
        """Rack-aware: 1st replica on one rack, 2nd and 3rd on another rack.""" 
        live = [d for d in self.datanodes if d.alive] 
        turn = next(self._rr) 
        first = live[turn % len(live)] 
        others = [d for d in live if d.rack != first.rack] 
        second = others[turn % len(others)] 
        third = [d for d in others if d.rack == second.rack and d is not second][ 
            0 
        ] 
        return [first, second, third] 
  
    def put(self, path, data): 
        ids = [] 
        for start in range(0, len(data), BLOCK_SIZE): 
            chunk = data[start : start + BLOCK_SIZE] 
            bid = f"blk_{self._next_id:04d}" 
            self._next_id += 1 
            targets = self.choose_targets() 
            for dn in targets:  # replication pipeline 
                dn.blocks[bid] = chunk 
            self.block_map[bid] = [dn.name for dn in targets] 
            ids.append(bid) 
        self.namespace[path] = ids 
        return ids 
  
    def get(self, path): 
        out = [] 
        for bid in self.namespace[path]: 
            for name in self.block_map[bid]: 
                dn = self.node(name) 
                if dn.alive:  # read from first live replica 
                    out.append(dn.blocks[bid]) 
                    break 
        return "".join(out) 
  
    def node(self, name): 
        return next(d for d in self.datanodes if d.name == name) 
  
    def handle_failure(self, name): 
        failed = self.node(name) 
        failed.alive = False 
        repaired = 0 
        for bid, holders in self.block_map.items(): 
            if name in holders: 
                holders.remove(name) 
                source = self.node(holders[0]) 
                spare = [ 
                    d for d in self.datanodes if d.alive and d.name not in holders 
                ][0] 
                spare.blocks[bid] = source.blocks[bid] 
                holders.append(spare.name) 
                repaired += 1 
        return repaired 
  
  
def checksum(text): 
    return hashlib.md5(text.encode()).hexdigest()[:8] 
  
  
cluster = [ 
    DataNode("DN1", "Rack-1"), 
    DataNode("DN2", "Rack-1"), 
    DataNode("DN3", "Rack-1"), 
    DataNode("DN4", "Rack-2"), 
    DataNode("DN5", "Rack-2"), 
    DataNode("DN6", "Rack-2"), 
] 
nn = NameNode(cluster) 
  
log = "\n".join( 
    f"2026-03-0{d} 10:{m:02d} sensor={s} temp={30 + (d * m + s) % 9}" 
    for d in range(1, 3) 
    for m in (5, 20) 
    for s in (1, 2) 
) 
  
print("Distributed File System Simulation (HDFS)") 
print("=" * 45) 
print(f"Block size = {BLOCK_SIZE} bytes | Replication factor = {REPLICATION}") 
print("Cluster    :", ", ".join(f"{d.name}({d.rack})" for d in cluster)) 
  
print("\n$ hdfs dfs -put sensor_log.txt /data/sensor_log.txt") 
blocks = nn.put("/data/sensor_log.txt", log) 
print(f"File size  = {len(log)} bytes  ->  {len(blocks)} blocks") 
  
print("\nNameNode metadata (block -> DataNodes):") 
for bid in blocks: 
    racks = {nn.node(n).rack for n in nn.block_map[bid]} 
    print(f"  {bid}  {nn.block_map[bid]}  racks={sorted(racks)}") 
  
print("\nDataNode storage report:") 
for d in cluster: 
    print(f"  {d.name}  blocks={sorted(d.blocks)}") 
  
print("\n$ hdfs dfs -cat /data/sensor_log.txt | head -3") 
data = nn.get("/data/sensor_log.txt") 
print("\n".join(data.splitlines()[:3])) 
print("Checksum original =", checksum(log), "| read back =", checksum(data)) 
  
print("\n*** DataNode DN1 fails (heartbeat lost) ***") 
fixed = nn.handle_failure("DN1") 
print("Under-replicated blocks re-replicated:", fixed) 
for bid in blocks: 
    print(f"  {bid}  {nn.block_map[bid]}") 
data2 = nn.get("/data/sensor_log.txt") 
print( 
    "File readable after failure:", data2 == log, "| checksum =", checksum(data2) 
) 
print("\n$ hdfs fsck /data/sensor_log.txt -files -blocks") 
healthy = all(len(v) == REPLICATION for v in nn.block_map.values()) 
print( 
f"Total blocks: {len(blocks)} | Replication: {REPLICATION} | " 
f"Status: {'HEALTHY' if healthy else 'CORRUPT'}" 
)

Distributed File System Simulation (HDFS)
Block size = 64 bytes | Replication factor = 3
Cluster    : DN1(Rack-1), DN2(Rack-1), DN3(Rack-1), DN4(Rack-2), DN5(Rack-2), DN6(Rack-2)

$ hdfs dfs -put sensor_log.txt /data/sensor_log.txt
File size  = 271 bytes  ->  5 blocks

NameNode metadata (block -> DataNodes):
  blk_0001  ['DN1', 'DN4', 'DN5']  racks=['Rack-1', 'Rack-2']
  blk_0002  ['DN2', 'DN5', 'DN4']  racks=['Rack-1', 'Rack-2']
  blk_0003  ['DN3', 'DN6', 'DN4']  racks=['Rack-1', 'Rack-2']
  blk_0004  ['DN4', 'DN1', 'DN2']  racks=['Rack-1', 'Rack-2']
  blk_0005  ['DN5', 'DN2', 'DN1']  racks=['Rack-1', 'Rack-2']

DataNode storage report:
  DN1  blocks=['blk_0001', 'blk_0004', 'blk_0005']
  DN2  blocks=['blk_0002', 'blk_0004', 'blk_0005']
  DN3  blocks=['blk_0003']
  DN4  blocks=['blk_0001', 'blk_0002', 'blk_0003', 'blk_0004']
  DN5  blocks=['blk_0001', 'blk_0002', 'blk_0005']
  DN6  blocks=['blk_0003']

$ hdfs dfs -cat /data/sensor_log.txt | head -3
2026-03-01 10:05 sensor=1 temp=36
20

# 14.Word Count using MapReduce (Map, Shuffle-Sort and Reduce Phases)

In [3]:
"""
Word count in Hadoop Streaming style. 
  Local simulation : python exp14.py 
  Hadoop Streaming : hadoop jar hadoop-streaming.jar \ 
                       -files exp14.py -mapper "python exp14.py map" \ 
                       -reducer "python exp14.py reduce" \ 
                       -input /data/news -output /out/wc 
  Unix pipe test   : cat news.txt | python exp14.py map | sort | python exp14.py reduce 
""" 
  
import re 
import sys 
from itertools import groupby 
  
  
def mapper(lines): 
    for line in lines: 
        for word in re.findall(r"[a-z]+", line.lower()): 
            yield f"{word}\t1" 
  
  
def reducer(lines): 
    pairs = (line.rstrip("\n").split("\t") for line in lines) 
    for word, group in groupby(pairs, key=lambda kv: kv[0]): 
        yield f"{word}\t{sum(int(c) for _, c in group)}" 
  
  
if len(sys.argv) > 1 and sys.argv[1] == "map": 
    for out in mapper(sys.stdin): 
        print(out) 
    sys.exit() 
if len(sys.argv) > 1 and sys.argv[1] == "reduce": 
    for out in reducer(sys.stdin): 
        print(out) 
    sys.exit() 
  
# ---------------- Local simulation of the Hadoop job ---------------- 
splits = { 
    "split-1 (node1)": [ 
        "Data science uses data to create value", 
        "Big data needs distributed storage", 
    ], 
    "split-2 (node2)": [ 
        "Hadoop stores big data in HDFS", 
        "MapReduce processes data in parallel", 
    ], 
    "split-3 (node3)": [ 
        "Spark and Hadoop process big data", 
        "Data drives science", 
    ], 
} 
  
print("MapReduce Word Count on News Headlines") 
print("=" * 42) 
  
print("\n--- MAP PHASE ---") 
mapped = {} 
for split, lines in splits.items(): 
    out = list(mapper(lines)) 
    mapped[split] = out 
    print( 
        f"{split}: {len(out)} pairs ->", 
        ", ".join(o.replace("\t", ":") for o in out[:6]), 
        "...", 
    ) 
  
print("\n--- COMBINER (local aggregation on each mapper) ---") 
combined = [] 
for split, out in mapped.items(): 
    local = list(reducer(sorted(out))) 
    combined.extend(local) 
    print(f"{split}: {len(out)} pairs reduced to {len(local)}") 
  
print("\n--- SHUFFLE AND SORT ---") 
shuffled = sorted(combined) 
for key, grp in groupby(shuffled, key=lambda s: s.split("\t")[0]): 
    values = [g.split("\t")[1] for g in grp] 
    if len(values) > 1: 
        print(f"  {key:<12} -> {values}") 
print("  (keys appearing on only one mapper are not shown)") 
  
print("\n--- REDUCE PHASE ---") 
result = [r.split("\t") for r in reducer(shuffled)] 
result.sort(key=lambda kv: (-int(kv[1]), kv[0])) 
print(f"{'WORD':<12}COUNT") 
for word, count in result: 
    print(f"{word:<12}{count}") 
  
print("\nDistinct words :", len(result)) 
print("Total words    :", sum(int(c) for _, c in result)) 
print("Top 3 words    :", [w for w, _ in result[:3]])

MapReduce Word Count on News Headlines

--- MAP PHASE ---
split-1 (node1): 12 pairs -> data:1, science:1, uses:1, data:1, to:1, create:1 ...
split-2 (node2): 11 pairs -> hadoop:1, stores:1, big:1, data:1, in:1, hdfs:1 ...
split-3 (node3): 9 pairs -> spark:1, and:1, hadoop:1, process:1, big:1, data:1 ...

--- COMBINER (local aggregation on each mapper) ---
split-1 (node1): 12 pairs reduced to 10
split-2 (node2): 11 pairs reduced to 9
split-3 (node3): 9 pairs reduced to 8

--- SHUFFLE AND SORT ---
  big          -> ['1', '1', '1']
  data         -> ['2', '2', '3']
  hadoop       -> ['1', '1']
  science      -> ['1', '1']
  (keys appearing on only one mapper are not shown)

--- REDUCE PHASE ---
WORD        COUNT
data        7
big         3
hadoop      2
in          2
science     2
and         1
create      1
distributed 1
drives      1
hdfs        1
mapreduce   1
needs       1
parallel    1
process     1
processes   1
spark       1
storage     1
stores      1
to          1
uses        1
v